In [1]:
import os
import pandas as pd
import mlflow

c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [3]:
os.chdir(r"C:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps")


In [4]:
%pwd

'C:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [5]:
import numpy as np

In [6]:
from dataclasses import dataclass
from pathlib import Path
@dataclass

class model_evalulation_config:
    sim_matrix_path: Path
    featured_df_path :Path 
    model_eval_metrics : Path
    ml_flow_tracking_uri :str
    ml_flow_experiment_name :str
    ml_flow_run_name :str



In [7]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [8]:
class Congif_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_evaluation(self) -> model_evalulation_config:
        config = self.config.model_evalvation

        create_dir([config.model_eval_metrics])

        return model_evalulation_config(
            sim_matrix_path        = (config.sim_matrix_path),
            featured_df_path       = (config.featured_df_path),
            model_eval_metrics = (config.model_eval_metrics),
            ml_flow_tracking_uri   = config.ml_flow_tracking_uri,
        ml_flow_experiment_name = config.ml_flow_experiment_name,
        ml_flow_run_name        = config.ml_flow_run_name
        ) 

In [ ]:
class model_evalulation_congif:

    def __init__(self, config):
        self.config     = config
        self.sim_matrix = None
        self.df         = None

    def load_artifacts(self):
        try:
            logger.info("=" * 20 + " Loading Artifacts STARTED " + "=" * 20)
            with open(self.config.sim_matrix_path, 'rb') as f:
                self.sim_matrix = pickle.load(f)
            with open(self.config.featured_df_path, 'rb') as f:
                self.df = pickle.load(f)
            logger.info("sim_matrix : %s", str(self.sim_matrix.shape))
            logger.info("df         : %s", str(self.df.shape))
            logger.info("=" * 20 + " Loading Artifacts COMPLETED " + "=" * 20)

        except Exception as e:
            logger.error("Load artifacts FAILED - %s", str(e))
            raise e

    def get_test_sample(self) -> pd.DataFrame:
        try:
            logger.info("=" * 20 + " Test Sample STARTED " + "=" * 20)

            test_set = (
                self.df.groupby("aesthetic", group_keys=False)
                       .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
                       .drop_duplicates()
                       .reset_index(drop=True)
            )

            logger.info("Sample size : %d", len(test_set))
            logger.info("=" * 20 + " Test Sample COMPLETED " + "=" * 20)
            return test_set

        except Exception as e:
            logger.error("Test sample FAILED - %s", str(e))
            raise e

    def get_relevant(self, product_id, threshold=0.4):
        scores = self.sim_matrix[product_id].copy()
        return [
            i for i, s in enumerate(scores)
            if s >= threshold and i != product_id
        ]

    def precision_at_k(self, top_idx, relevant_idx, k) -> float:
        hits = len(set(top_idx[:k]) & set(relevant_idx))
        return hits / k if k > 0 else 0.0

    def ndcg_at_k(self, top_idx, relevant_idx, k) -> float:
        relevant_set = set(relevant_idx)
        dcg  = sum(
            1 / np.log2(i + 2)
            for i, idx in enumerate(top_idx[:k])
            if idx in relevant_set
        )
        idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(relevant_set))))
        return dcg / idcg if idcg > 0 else 0.0

    def map_score(self, top_idx, relevant_idx, k) -> float:
        relevant_set = set(relevant_idx)
        hits, score  = 0, 0.0
        for i, idx in enumerate(top_idx[:k]):
            if idx in relevant_set:
                hits  += 1
                score += hits / (i + 1)
        return score / len(relevant_set) if relevant_set else 0.0

    def score_product(self, row, top_k=10, threshold=0.5) -> dict:
        try:
            product_id   = row.name
            relevant_idx = self.get_relevant(product_id, threshold)

            scores             = self.sim_matrix[product_id].copy()
            scores[product_id] = -1
            top_idx            = scores.argsort()[::-1][0:top_k]

            precision = self.precision_at_k(top_idx, relevant_idx, top_k)
            ndcg      = self.ndcg_at_k(top_idx,      relevant_idx, top_k)
            map_s     = self.map_score(top_idx,       relevant_idx, top_k)

            logger.info(
                "%-35s | P@K=%.2f | NDCG=%.2f | MAP=%.2f",
                row["product_name_clean"][:35],
                precision, ndcg, map_s
            )

            return {
                "product_name_clean" : row["product_name_clean"],
                "aesthetic"          : row["aesthetic"],
                "n_relevant"         : len(relevant_idx),
                "precision_at_k"     : round(precision, 4),
                "ndcg_at_k"          : round(ndcg,      4),
                "map"                : round(map_s,     4),
            }

        except Exception as e:
            logger.error("score_product FAILED - %s", str(e))
            raise e

    def run_evaluation(self, top_k=10, threshold=0.4):
        try:
            logger.info("=" * 20 + " Evaluation STARTED " + "=" * 20)

            if self.df is None or self.sim_matrix is None:
                self.load_artifacts()

            test_sample = self.get_test_sample()
            logger.info("Scoring %d products", len(test_sample))

            records = []
            for _, row in test_sample.iterrows():
                result = self.score_product(row, top_k=top_k, threshold=threshold)
                records.append(result)

            metrics_df = pd.DataFrame(records)

            summary = {
                "n_products_tested" : len(metrics_df),
                "top_k"             : top_k,
                "threshold"         : threshold,
                "precision_at_k"    : round(metrics_df["precision_at_k"].mean(), 4),
                "ndcg_at_k"         : round(metrics_df["ndcg_at_k"].mean(),      4),
                "map"               : round(metrics_df["map"].mean(),             4),
            }

            out_path = os.path.join(
                self.config.model_eval_metrics, 'eval_metrics.csv'
            )
            metrics_df.to_csv(out_path, index=False)

            # ── MLflow ───────────────────────────────────
            mlflow.set_tracking_uri(self.config.ml_flow_tracking_uri)
            mlflow.set_experiment(self.config.ml_flow_experiment_name)

            with mlflow.start_run(run_name=self.config.ml_flow_run_name) as run:

                mlflow.log_param("top_k",     top_k)
                mlflow.log_param("threshold", threshold)
                mlflow.log_param("n_tested",  len(metrics_df))

                mlflow.log_metric("precision_at_k", summary["precision_at_k"])
                mlflow.log_metric("ndcg_at_k",      summary["ndcg_at_k"])
                mlflow.log_metric("map",            summary["map"])

                mlflow.log_artifact(out_path)

                mlflow.log_artifact(self.config.sim_matrix_path,  artifact_path="model")
                mlflow.log_artifact(self.config.featured_df_path, artifact_path="model")

                run_id        = run.info.run_id
                experiment_id = run.info.experiment_id

            # ── outside with block ───────────────────────
            model_uri = f"runs:/{run_id}/model"

            #client = MlflowClient()

            result = mlflow.register_model(
                model_uri=model_uri,
                name="FashionRecommendationModel"
            )

            logger.info("Model registered: %s Version: %s", result.name, result.version)

            run_url = f"{self.config.ml_flow_tracking_uri}/#/experiments/{experiment_id}/runs/{run_id}"

            # ── log summary ──────────────────────────────
            logger.info("======= SUMMARY =======")
            for k, v in summary.items():
                logger.info("%-25s : %s", k, v)
            logger.info("Run ID  : %s", run_id)
            logger.info("Run URL : %s", run_url)
            logger.info("=" * 20 + " Evaluation COMPLETED " + "=" * 20)

            return summary, metrics_df, run_id, run_url

        except Exception as e:
            logger.error("Evaluation FAILED - %s", str(e))
            raise e


In [10]:
con   = Congif_manager()
model = con.get_model_evaluation()
model = model_evalulation_congif(model)
model.run_evaluation()   



[2026-03-06 23:35:08,387: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-06 23:35:08,397: INFO: common: created directory at: artifacts]
[2026-03-06 23:35:08,402: INFO: common: created directory at: artifacts/model_evalvation/model/]
[2026-03-06 23:35:08,405: INFO: 2955397849: ==================== Evaluation STARTED ====================]
[2026-03-06 23:35:08,410: INFO: 2955397849: ==================== Loading Artifacts STARTED ====================]
[2026-03-06 23:35:20,047: INFO: 2955397849: sim_matrix : (11951, 11951)]
[2026-03-06 23:35:20,060: INFO: 2955397849: df         : (11951, 16)]
[2026-03-06 23:35:20,060: INFO: 2955397849: ==================== Loading Artifacts COMPLETED ====================]
[2026-03-06 23:35:20,060: INFO: 2955397849: ==================== Test Sample STARTED ====================]
[2026-03-06 23:35:20,164: INFO: 2955397849: Sample size : 35]
[2026-03-06 23:35:20,167: INFO: 2955397849: ==================== Test Sample COMPLETED ======

C:\Users\sam\AppData\Local\Temp\ipykernel_3496\2955397849.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(5, len(x)), random_state=42))


🏃 View run evaluation at: http://localhost:5000/#/experiments/1/runs/91647bd7b7294c1e9da4a2a88a1d02f0
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-03-06 23:35:24,563: ERROR: 2955397849: Evaluation FAILED - Unable to find a logged_model with artifact_path model under run 91647bd7b7294c1e9da4a2a88a1d02f0]


Successfully registered model 'FashionRecommendationModel'.


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_3496\2526157093.py", line 4, in <module>
    model.run_evaluation()
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_3496\2955397849.py", line 180, in run_evaluation
    raise e
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_3496\2955397849.py", line 159, in run_evaluation
    result = mlflow.register_model(
             ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\mlflow\tracking\_model_registry\fluent.py", line 136, in register_model
    return _register_model(
           ^^^^^^^^^^^^^^^^
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\mlflow\tracking\_model_registry\fluen

In [19]:
%pwd

'C:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [22]:
import pandas as pd 
with open(r'artifacts\feature_engineering\featured_data\featured_data.pkl','rb') as f:
   dff=  pickle.load(f)
   dff

In [15]:
import pickle

with open(r'artifacts\feature_engineering\model\sim_matrix.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)
    sim_matrix

In [24]:
dff.loc[1,'product_name']

"Men's Wonder-13 Sports Running Shoes…"

In [27]:
sim_matrix[54].argsort()[::-1][0:10]

array([  54, 2114, 8453, 2172,  106, 8440,  163, 8580,  258, 8539])

In [73]:
df[(df["aesthetic"] == 'streetwear')].index[1::][1:10]

Index([2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')

In [72]:
sim_matrix[1].argsort()[::-1][1:10]

array([8421, 2095,    1,   43, 8428,    5, 8426, 7354, 8435])

In [65]:
relevant_idx = df[
    (df["aesthetic"] == aesthetic) &
    (df.index != product_id)
].index.tolist()

In [66]:
print("relevant_idx count :", len(relevant_idx))
print("first 10           :", relevant_idx[:10])

relevant_idx count : 2581
first 10           : [3946, 3947, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955]


In [ ]:
dff.loc[41,'product_name']

np.float64(0.7465888640574219)

In [ ]:
import pickle
import pandas as pd

df = pickle.load(open('artifacts/feature_engineering/featured_data/featured_data.pkl', 'rb'))

print(df.columns.tolist())
print(df.shape)
print(df.head(2))